# Counter-flow heat exchanger — design & rating with the ε-NTU method

**Author:** Esteban López · Industrial Engineer  
**Tools:** ε-NTU & LMTD methods, CoolProp (real fluid properties), Python.

## Problem
Size and rate a **counter-flow water-to-water heat exchanger** that cools a hot process stream. Given the inlet
conditions and the overall heat-transfer coefficient, find the heat duty, outlet temperatures and effectiveness,
cross-check with LMTD, and produce the **design curve** (required area for a target outlet temperature).

> Self-contained and **reproducible**. Fluid properties come from **CoolProp**; if it is not installed, the
notebook falls back to constant water properties (the results barely change for this temperature range).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (9, 5), 'font.size': 11})
NAVY, CYAN, GOLD, GREY = '#0b1f3a', '#16b3c7', '#f2c14e', '#5b6b7f'

def cp_water(T_C):
    """Specific heat of water [J/kg.K] at temperature T [C], via CoolProp or a constant fallback."""
    try:
        from CoolProp.CoolProp import PropsSI
        return PropsSI('C', 'T', T_C + 273.15, 'P', 2e5, 'Water')
    except Exception:
        return 4180.0

## 1. Inputs and stream capacity rates

In [ ]:
Th_in, Tc_in = 90.0, 20.0      # inlet temperatures [C]
mh, mc = 2.0, 2.5              # mass flow rates [kg/s]
U, A = 1200.0, 8.0            # overall coeff [W/m2.K], area [m2]

cph = cp_water((Th_in + 55) / 2)   # evaluate cp near each stream's mean T
cpc = cp_water((Tc_in + 50) / 2)
Ch, Cc = mh * cph, mc * cpc        # capacity rates [W/K]
Cmin, Cmax = min(Ch, Cc), max(Ch, Cc)
Cr = Cmin / Cmax
print(f'cp_hot={cph:.0f}  cp_cold={cpc:.0f} J/kg.K')
print(f'C_hot={Ch:.0f}  C_cold={Cc:.0f} W/K  ->  C_min={Cmin:.0f}, Cr={Cr:.3f}')

## 2. Rating with ε-NTU
For a counter-flow exchanger:  ε = (1 − e^(−NTU(1−Cr))) / (1 − Cr·e^(−NTU(1−Cr)))

In [ ]:
def eps_counterflow(NTU, Cr):
    if abs(Cr - 1) < 1e-9:
        return NTU / (1 + NTU)
    e = np.exp(-NTU * (1 - Cr))
    return (1 - e) / (1 - Cr * e)

NTU = U * A / Cmin
eps = eps_counterflow(NTU, Cr)
Q = eps * Cmin * (Th_in - Tc_in)
Th_out = Th_in - Q / Ch
Tc_out = Tc_in + Q / Cc
print(f'NTU = {NTU:.3f}   effectiveness ε = {eps*100:.1f} %')
print(f'Heat duty Q = {Q/1000:.1f} kW')
print(f'Hot  outlet = {Th_out:.1f} °C   Cold outlet = {Tc_out:.1f} °C')

## 3. Independent cross-check with LMTD
If the ε-NTU and LMTD methods agree, the calculation is consistent.

In [ ]:
dT1 = Th_in - Tc_out       # counter-flow terminal differences
dT2 = Th_out - Tc_in
LMTD = (dT1 - dT2) / np.log(dT1 / dT2)
Q_lmtd = U * A * LMTD
print(f'LMTD = {LMTD:.2f} °C   Q(LMTD) = {Q_lmtd/1000:.1f} kW   Q(ε-NTU) = {Q/1000:.1f} kW')
print('Agreement:', 'OK' if abs(Q_lmtd - Q) / Q < 0.01 else 'CHECK')

## 4. ε-NTU design chart with the operating point

In [ ]:
NT = np.linspace(0, 6, 300)
fig, ax = plt.subplots()
for cr in [0, 0.25, 0.5, 0.75, 1.0]:
    ax.plot(NT, [eps_counterflow(n, cr)*100 for n in NT], label=f'Cr={cr}')
ax.plot(NTU, eps*100, 'o', color=GOLD, ms=11)
ax.annotate(f'design\nNTU={NTU:.2f}, ε={eps*100:.0f}%', (NTU, eps*100),
            xytext=(NTU+0.6, eps*100-15), color='#b8902a', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=GOLD))
ax.set_xlabel('NTU = UA / C_min'); ax.set_ylabel('Effectiveness ε (%)')
ax.set_title('ε-NTU design chart — counter-flow', fontweight='bold', color=NAVY)
ax.legend(title='Capacity ratio'); ax.grid(alpha=.25); plt.tight_layout(); plt.show()

## 5. Temperature profile along the exchanger
Integrating the local energy balance (dT/da = ±U·ΔT/C) along the area, from the hot-inlet end.

In [ ]:
n = 400; a = np.linspace(0, A, n); da = a[1]-a[0]
Th = np.zeros(n); Tc = np.zeros(n); Th[0] = Th_in; Tc[0] = Tc_out
for i in range(n-1):
    d = Th[i] - Tc[i]
    Th[i+1] = Th[i] - U*d/Ch*da
    Tc[i+1] = Tc[i] - U*d/Cc*da
fig, ax = plt.subplots()
ax.plot(a, Th, color=GOLD, lw=2.5, label=f'Hot  {Th_in:.0f}→{Th[-1]:.0f} °C')
ax.plot(a, Tc, color=CYAN, lw=2.5, label=f'Cold {Tc[-1]:.0f}→{Tc[0]:.0f} °C')
ax.fill_between(a, Th, Tc, color=GREY, alpha=.12)
ax.set_xlabel('Area a (m²)'); ax.set_ylabel('Temperature (°C)')
ax.set_title('Counter-flow temperature profile', fontweight='bold', color=NAVY)
ax.legend(); ax.grid(alpha=.25); plt.tight_layout(); plt.show()

## 6. Design curve — required area for a target hot outlet
Invert the problem: for a target hot-outlet temperature, find the required effectiveness, NTU and area.

In [ ]:
from scipy.optimize import brentq
def area_for_target(Th_target):
    eps_req = (Th_in - Th_target) * Ch / (Cmin * (Th_in - Tc_in))
    if eps_req >= 1: return np.nan
    NTU_req = brentq(lambda N: eps_counterflow(N, Cr) - eps_req, 1e-6, 50)
    return NTU_req * Cmin / U

targets = np.arange(45, 71, 2.0)
areas = [area_for_target(t) for t in targets]
fig, ax = plt.subplots()
ax.plot(targets, areas, color=CYAN, lw=2.5, marker='o')
ax.axvline(Th_out, color=GOLD, ls='--'); ax.axhline(A, color=GOLD, ls='--')
ax.set_xlabel('Target hot-outlet temperature (°C)'); ax.set_ylabel('Required area (m²)')
ax.set_title('Design curve — colder outlet costs disproportionately more area', fontweight='bold', color=NAVY)
ax.grid(alpha=.25); plt.tight_layout(); plt.show()
print('To cool the hot stream to 45 °C you would need ≈ %.1f m² (vs %.0f m² now).' % (area_for_target(45), A))

## 7. Conclusion

- The 8 m² counter-flow exchanger delivers **ε ≈ 56 %** and **Q ≈ 330 kW**, cooling the hot stream from 90 °C to ≈ 51 °C. ε-NTU and LMTD agree to <1 %.
- The **design curve** shows diminishing returns: pushing the outlet colder than ~50 °C costs disproportionately more area, because effectiveness flattens at high NTU (see the ε-NTU chart).
- **Engineering decisions** this enables: pick area/U for a target duty, judge whether counter-flow is worth it vs parallel-flow, and quantify the penalty of fouling (a lower U shifts the operating point left on the ε-NTU chart).

*Extensions:* add real pressure-drop and pumping-power trade-off, fouling factor over time, and a parallel-flow / cross-flow comparison.